In [ ]:
from google.colab import files

uploaded = files.upload()  # Select all 3–5 raw_data files at once
print("Uploaded files:", list(uploaded.keys()))

Saving combine.csv to combine (1).csv
Saving cleaned_data (2).csv to cleaned_data (2).csv
Uploaded files: ['combine (1).csv', 'cleaned_data (2).csv']


In [ ]:
import pandas as pd
import re

# ── LOAD ALL FILES ────────────────────────────────────────────────────
df_auni   = pd.read_csv('mudah_auni.csv',      low_memory=False)
df_goe    = pd.read_csv('mudah_goe.csv',        low_memory=False)
df_yasmin = pd.read_csv('mudah_yasmin (1).csv', low_memory=False)

# ── FIX YASMIN: missing seller_name column ────────────────────────────
if 'seller_name' not in df_yasmin.columns:
    df_yasmin['seller_name'] = None

# ── FIX GOE: seller_name has some URLs leaked in (scraper bug) ────────
if 'seller_name' in df_goe.columns:
    url_mask = df_goe['seller_name'].astype(str).str.startswith('http', na=False)
    df_goe.loc[url_mask, 'seller_name'] = None

# ── ENFORCE DTYPES ON ALL 3 FILES BEFORE CONCAT ───────────────────────
def fix_dtypes(df, source_name):
    print(f"\n── Fixing {source_name} ──")

    # listing_id → string (safest common type, no float conversion)
    df['listing_id'] = df['listing_id'].astype(str).str.strip()
    df['listing_id'] = df['listing_id'].str.replace(r'\.0$', '', regex=True)  # remove ".0" suffix

    # price_rm → numeric
    df['price_rm'] = pd.to_numeric(
        df['price_rm'].astype(str).str.replace(r'[^\d.]', '', regex=True),
        errors='coerce'
    )

    # size_sqft → numeric
    df['size_sqft'] = pd.to_numeric(df['size_sqft'], errors='coerce')

    # bedrooms → numeric, cap at 20
    df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
    df.loc[df['bedrooms'] > 20, 'bedrooms'] = None

    # bathrooms → numeric, cap at 20
    df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')
    df.loc[df['bathrooms'] > 20, 'bathrooms'] = None

    # mortgage_est_rm → numeric
    df['mortgage_est_rm'] = pd.to_numeric(df['mortgage_est_rm'], errors='coerce')

    # All text columns → clean string, empty → None
    text_cols = ['property_type', 'description', 'location', 'state',
                 'land_title', 'tenure', 'seller_type', 'seller_name',
                 'url', 'img_url']
    for col in text_cols:
        if col in df.columns:
            df[col] = (df[col].astype(str)
                               .str.strip()
                               .replace({'nan': None, 'None': None, '': None}))

    print(f"  Shape     : {df.shape}")
    print(f"  Dtypes OK : listing_id={df['listing_id'].dtype}, "
          f"price_rm={df['price_rm'].dtype}, "
          f"bedrooms={df['bedrooms'].dtype}, "
          f"bathrooms={df['bathrooms'].dtype}")
    return df

df_auni   = fix_dtypes(df_auni,   'AUNI')
df_goe    = fix_dtypes(df_goe,    'GOE')
df_yasmin = fix_dtypes(df_yasmin, 'YASMIN')

# ── CONCAT ────────────────────────────────────────────────────────────
raw_df = pd.concat([df_auni, df_goe, df_yasmin], ignore_index=True)

# ── FINAL DTYPE CHECK ─────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"  Combined shape : {raw_df.shape}")
print(f"  Dtypes:\n{raw_df.dtypes}")
print(f"\n  Bedrooms  null : {raw_df['bedrooms'].isna().sum():,}")
print(f"  Bathrooms null : {raw_df['bathrooms'].isna().sum():,}")
print(f"  Price_rm  null : {raw_df['price_rm'].isna().sum():,}")
print(f"{'='*50}")

raw_df.head()


── Fixing AUNI ──
  Shape     : (31393, 16)
  Dtypes OK : listing_id=object, price_rm=int64, bedrooms=float64, bathrooms=float64

── Fixing GOE ──
  Shape     : (39348, 16)
  Dtypes OK : listing_id=object, price_rm=float64, bedrooms=float64, bathrooms=float64

── Fixing YASMIN ──
  Shape     : (30678, 16)
  Dtypes OK : listing_id=object, price_rm=int64, bedrooms=float64, bathrooms=float64

  Combined shape : (101419, 16)
  Dtypes:
listing_id          object
property_type       object
description         object
location            object
state               object
price_rm           float64
mortgage_est_rm    float64
land_title          object
size_sqft          float64
bedrooms           float64
bathrooms          float64
tenure              object
seller_type         object
seller_name         object
url                 object
img_url             object
dtype: object

  Bedrooms  null : 18,259
  Bathrooms null : 18,183
  Price_rm  null : 40


,listing_id,property_type,description,location,state,price_rm,mortgage_est_rm,land_title,size_sqft,bedrooms,bathrooms,tenure,seller_type,seller_name,url,img_url
0,114694294,New Service Residenceforsale,"200 meter to KLCC , 2R2B Dual Key Option (Faci...",KLCC,Kuala Lumpur,700000.0,2789.0,Non Bumi Lot,700.0,2.0,2.0,Leasehold,Agent,None,https://www.mudah.my/200-meter-to-klcc-2r2b-du...,https://cdn.rnudah.com/images/plain/479a15de7b...
1,112387490,Apartmentforsale,[❤️Booking1k❤️CashBack150k❤️FullLoan❤️]Teratai...,"Teratai Mewah Apartment Block 4 & 6, Setapak",Kuala Lumpur,185000.0,737.0,Non Bumi Lot,650.0,3.0,1.0,Freehold,Agent,None,https://www.mudah.my/booking1k-cashback150k-fu...,https://cdn.rnudah.com/images/plain/351e2259f4...
2,114152305,New Apartmentforsale,KL Sungai Besi Fully Residential Condo\n📍 Band...,"PSV 1 Residences @ Platinum South Valley, Band...",Kuala Lumpur,410000.0,1634.0,Malay Reserved,1376.0,4.0,3.0,Leasehold,Agent,None,https://www.mudah.my/kl-sungai-besi-fully-resi...,https://cdn.rnudah.com/images/plain/bc635fd67b...
3,110424536,Zero-Lot Bungalowforsale,[FULLY FURNISHED] Bungalow 16 Quartz Taman Mel...,Taman Melawati,Kuala Lumpur,3100000.0,12353.0,Non Bumi Lot,5270.0,5.0,6.0,Leasehold,Agent,None,https://www.mudah.my/fully-furnished-bungalow-...,https://cdn.rnudah.com/images/plain/9c3b1f2c46...
4,114199319,1-storey Terraced Houseforsale,END LOT RENOVATED Single Storey Sri Damansara ...,Sri Damansara,Kuala Lumpur,750000.0,2989.0,Non Bumi Lot,1400.0,4.0,3.0,Freehold,Agent,None,https://www.mudah.my/end-lot-renovated-single-...,https://cdn.rnudah.com/images/plain/e92246e237...


In [ ]:
print("=== Shape ===")
print(raw_df.shape)

print("\n=== Columns & Types ===")
print(raw_df.dtypes)

print("\n=== Missing Values ===")
print(raw_df.isnull().sum())

print("\n=== Duplicate Rows ===")
print(raw_df.duplicated().sum())

print("\n=== Sample ===")
raw_df.head(3)

=== Shape ===
(101419, 16)

=== Columns & Types ===
listing_id          object
property_type       object
description         object
location            object
state               object
price_rm           float64
mortgage_est_rm    float64
land_title          object
size_sqft          float64
bedrooms           float64
bathrooms          float64
tenure              object
seller_type         object
seller_name         object
url                 object
img_url             object
dtype: object

=== Missing Values ===
listing_id              0
property_type          29
description            61
location               35
state                  36
price_rm               40
mortgage_est_rm       801
land_title             65
size_sqft           10509
bedrooms            18259
bathrooms           18183
tenure                 40
seller_type            37
seller_name        100646
url                    40
img_url                47
dtype: int64

=== Duplicate Rows ===
9134

=== Sample ===


,listing_id,property_type,description,location,state,price_rm,mortgage_est_rm,land_title,size_sqft,bedrooms,bathrooms,tenure,seller_type,seller_name,url,img_url
0,114694294,New Service Residenceforsale,"200 meter to KLCC , 2R2B Dual Key Option (Faci...",KLCC,Kuala Lumpur,700000.0,2789.0,Non Bumi Lot,700.0,2.0,2.0,Leasehold,Agent,None,https://www.mudah.my/200-meter-to-klcc-2r2b-du...,https://cdn.rnudah.com/images/plain/479a15de7b...
1,112387490,Apartmentforsale,[❤️Booking1k❤️CashBack150k❤️FullLoan❤️]Teratai...,"Teratai Mewah Apartment Block 4 & 6, Setapak",Kuala Lumpur,185000.0,737.0,Non Bumi Lot,650.0,3.0,1.0,Freehold,Agent,None,https://www.mudah.my/booking1k-cashback150k-fu...,https://cdn.rnudah.com/images/plain/351e2259f4...
2,114152305,New Apartmentforsale,KL Sungai Besi Fully Residential Condo\n📍 Band...,"PSV 1 Residences @ Platinum South Valley, Band...",Kuala Lumpur,410000.0,1634.0,Malay Reserved,1376.0,4.0,3.0,Leasehold,Agent,None,https://www.mudah.my/kl-sungai-besi-fully-resi...,https://cdn.rnudah.com/images/plain/bc635fd67b...


In [ ]:
import re

cleaned_df = raw_df.copy()

# ── 1. STANDARDIZE COLUMN NAMES ──────────────────────────────────────
cleaned_df.columns = (
    cleaned_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace(r'[^\w]', '', regex=True)
)

# ── 2. REMOVE DUPLICATE ROWS ─────────────────────────────────────────
before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates()
print(f"Duplicates removed: {before - len(cleaned_df)}")

# ── 3. STRIP WHITESPACE & NORMALIZE TEXT COLUMNS ─────────────────────
str_cols = cleaned_df.select_dtypes(include='object').columns

for col in str_cols:
    cleaned_df[col] = cleaned_df[col].str.strip()
    cleaned_df[col] = cleaned_df[col].str.replace(r'\s+', ' ', regex=True)  # collapse multiple spaces

# ── 4. REMOVE HTML TAGS (common in scraped data) ──────────────────────
def strip_html(text):
    if pd.isna(text):
        return text
    return re.sub(r'<[^>]+>', '', str(text)).strip()

# Apply to all text columns — skip URL columns
non_url_cols = [c for c in str_cols if 'url' not in c and 'link' not in c and 'href' not in c]
for col in non_url_cols:
    cleaned_df[col] = cleaned_df[col].apply(strip_html)

# ── 5. CLEAN & VALIDATE URLs ─────────────────────────────────────────
url_cols = [c for c in cleaned_df.columns if 'url' in c or 'link' in c or 'href' in c]

for col in url_cols:
    # Remove rows with clearly broken URLs
    cleaned_df[col] = cleaned_df[col].str.strip()
    invalid_mask = ~cleaned_df[col].str.startswith(('http://', 'https://'), na=False)
    print(f"  '{col}' — invalid/missing URLs: {invalid_mask.sum()}")
    cleaned_df.loc[invalid_mask, col] = None   # set to null, don't drop yet

# ── 6. HANDLE MISSING VALUES ─────────────────────────────────────────
print("\nMissing before:\n", cleaned_df.isnull().sum())

# Drop rows where ALL values are null
cleaned_df = cleaned_df.dropna(how='all')

# For text/scraped content columns — fill with 'N/A'
for col in non_url_cols:
    cleaned_df[col] = cleaned_df[col].fillna('N/A')

# For numeric columns — fill with median
num_cols = cleaned_df.select_dtypes(include='number').columns
for col in num_cols:
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

print("\nMissing after:\n", cleaned_df.isnull().sum())

# ── 7. FIX DATA TYPES ────────────────────────────────────────────────
# Dates — adjust column name to match yours
date_cols = [c for c in cleaned_df.columns if 'date' in c or 'time' in c or 'crawled' in c]
for col in date_cols:
    cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')
    print(f"  '{col}' converted to datetime")

# Numbers stored as strings (e.g. price, count, rating)
num_str_cols = [c for c in cleaned_df.columns if any(k in c for k in ['price', 'count', 'rating', 'score', 'views', 'num'])]
for col in num_str_cols:
    cleaned_df[col] = (
        cleaned_df[col].astype(str)
        .str.replace(r'[^\d.]', '', regex=True)   # remove $, commas, etc.
        .replace('', None)
        .astype(float)
    )
    print(f"  '{col}' converted to float")


# ── 8. RESET INDEX ────────────────────────────────────────────────────
cleaned_df = cleaned_df.reset_index(drop=True)

print(f"\n✅ Final cleaned shape: {cleaned_df.shape}")
cleaned_df.head()

Duplicates removed: 9134
  'url' — invalid/missing URLs: 40
  'img_url' — invalid/missing URLs: 47

Missing before:
 listing_id             0
property_type         29
description           61
location              35
state                 36
price_rm              40
mortgage_est_rm      785
land_title            65
size_sqft           9399
bedrooms           16286
bathrooms          16212
tenure                40
seller_type           37
seller_name        91512
url                   40
img_url               47
dtype: int64

Missing after:
 listing_id          0
property_type       0
description         0
location            0
state               0
price_rm            0
mortgage_est_rm     0
land_title          0
size_sqft           0
bedrooms            0
bathrooms           0
tenure              0
seller_type         0
seller_name         0
url                40
img_url            47
dtype: int64
  'price_rm' converted to float

✅ Final cleaned shape: (92285, 16)


,listing_id,property_type,description,location,state,price_rm,mortgage_est_rm,land_title,size_sqft,bedrooms,bathrooms,tenure,seller_type,seller_name,url,img_url
0,114694294,New Service Residenceforsale,"200 meter to KLCC , 2R2B Dual Key Option (Faci...",KLCC,Kuala Lumpur,700000.0,2789.0,Non Bumi Lot,700.0,2.0,2.0,Leasehold,Agent,N/A,https://www.mudah.my/200-meter-to-klcc-2r2b-du...,https://cdn.rnudah.com/images/plain/479a15de7b...
1,112387490,Apartmentforsale,[❤️Booking1k❤️CashBack150k❤️FullLoan❤️]Teratai...,"Teratai Mewah Apartment Block 4 & 6, Setapak",Kuala Lumpur,185000.0,737.0,Non Bumi Lot,650.0,3.0,1.0,Freehold,Agent,N/A,https://www.mudah.my/booking1k-cashback150k-fu...,https://cdn.rnudah.com/images/plain/351e2259f4...
2,114152305,New Apartmentforsale,KL Sungai Besi Fully Residential Condo 📍 Banda...,"PSV 1 Residences @ Platinum South Valley, Band...",Kuala Lumpur,410000.0,1634.0,Malay Reserved,1376.0,4.0,3.0,Leasehold,Agent,N/A,https://www.mudah.my/kl-sungai-besi-fully-resi...,https://cdn.rnudah.com/images/plain/bc635fd67b...
3,110424536,Zero-Lot Bungalowforsale,[FULLY FURNISHED] Bungalow 16 Quartz Taman Mel...,Taman Melawati,Kuala Lumpur,3100000.0,12353.0,Non Bumi Lot,5270.0,5.0,6.0,Leasehold,Agent,N/A,https://www.mudah.my/fully-furnished-bungalow-...,https://cdn.rnudah.com/images/plain/9c3b1f2c46...
4,114199319,1-storey Terraced Houseforsale,END LOT RENOVATED Single Storey Sri Damansara ...,Sri Damansara,Kuala Lumpur,750000.0,2989.0,Non Bumi Lot,1400.0,4.0,3.0,Freehold,Agent,N/A,https://www.mudah.my/end-lot-renovated-single-...,https://cdn.rnudah.com/images/plain/e92246e237...


Validation report

In [ ]:
print("=" * 40)
print("       CLEANING SUMMARY REPORT")
print("=" * 40)
print(f"Raw rows      : {len(raw_df)}")
print(f"Cleaned rows  : {len(cleaned_df)}")
print(f"Rows removed  : {len(raw_df) - len(cleaned_df)}")
print(f"\nColumns       : {cleaned_df.columns.tolist()}")
print(f"\nRemaining nulls:\n{cleaned_df.isnull().sum()}")
print(f"\nData types:\n{cleaned_df.dtypes}")

       CLEANING SUMMARY REPORT
Raw rows      : 101419
Cleaned rows  : 92285
Rows removed  : 9134

Columns       : ['listing_id', 'property_type', 'description', 'location', 'state', 'price_rm', 'mortgage_est_rm', 'land_title', 'size_sqft', 'bedrooms', 'bathrooms', 'tenure', 'seller_type', 'seller_name', 'url', 'img_url']

Remaining nulls:
listing_id          0
property_type       0
description         0
location            0
state               0
price_rm            0
mortgage_est_rm     0
land_title          0
size_sqft           0
bedrooms            0
bathrooms           0
tenure              0
seller_type         0
seller_name         0
url                40
img_url            47
dtype: int64

Data types:
listing_id          object
property_type       object
description         object
location            object
state               object
price_rm           float64
mortgage_est_rm    float64
land_title          object
size_sqft          float64
bedrooms           float64
bathrooms  

Save & download

In [ ]:
cleaned_df.to_csv('cleaned_data.csv', index=False)
print("✅ cleaned_data.csv saved!")

# Download both files
from google.colab import files
files.download('cleaned_data.csv')

KeyboardInterrupt: 